# 🌊 BinWaves Duke (Validation - All Buoys)


> **Inputs:**
> - `Kp_coefficients.nc` file saved in the `outputs/` folder, obtained with the `Propagation Notebook`.
> - OPTIONAL: `buoy_buoy_ID_bulk_parameters.pkl`: wave bulk parameters (Hs, Tp, Dp) stored in the `inputs/buoy_data/` folder.
>
>
> **Outputs:**
> - Plots for each buoy if needed.
> - CSV files with validation data for each buoy saved in `outputs/time_series/` folder.


### BinWaves Validation - All Buoys

This notebook processes **all available buoys** for validation, looping through each one automatically.

**Key Features:**
- Processes all buoys in the `buoys` dictionary
- Handles missing variables gracefully (plots only what's available)
- Saves plots and CSV files for each buoy
- Skips buoys that fail to load (with error messages)


In [1]:
import xarray as xr
import numpy as np
import pandas as pd
import sys
import os
from pathlib import Path

# Add utils to path
sys.path.append('../utils')
from operations import transform_Offshore_spectrum
from plotting import plot_wave_series
from bluemath_tk.waves.binwaves import reconstruc_spectra

# Load CAWCR spectrum
cawcr_spectrum = xr.open_dataset(
    "inputs/440141904108_spec_WHACS.nc"
)
print("CAWCR spectrum loaded")
cawcr_spectrum


CAWCR spectrum loaded


<xarray.Dataset> Size: 2GB
Dimensions:    (time: 394464, freq: 29, dir: 24)
Coordinates:
  * time       (time) datetime64[ns] 3MB 1979-01-01 ... 2023-12-31T23:00:00
  * freq       (freq) float64 232B 0.035 0.0385 0.04235 ... 0.4171 0.4589 0.5047
  * dir        (dir) float64 192B 7.5 22.5 37.5 52.5 ... 307.5 322.5 337.5 352.5
    seapoint   int32 4B ...
    station    int64 8B ...
Data variables:
    efth       (time, freq, dir) float64 2GB ...
    latitude   float64 8B ...
    longitude  float64 8B ...

In [2]:
# Define all buoys and their coordinates
buoys = {
     "44095": (-75.3300, 35.7500),
     "41025": (-75.4540, 35.0100),
     # "41120": (-75.2580, 35.2580),
  
     # "dsln7": (-75.2970, 35.1530), 
     # "41017": (-75.1000, 35.4000),
     # "41015": (-75.3000, 35.4000),
     # "44100": (-75.5930, 36.2580),
     # "jprn7": (-75.5870, 35.9120),
     # "44056": (-75.7140, 36.2000),
     # "44086": (-75.3300, 35.7500),
    
     # "44006": (-75.4000, 36.3000),
     # "44079": (-75.5930, 36.1750),
     # "44019": (-75.2000, 36.4000),
}

print(f"Total buoys to process: {len(buoys)}")
print("Buoy IDs:", list(buoys.keys()))


Total buoys to process: 2
Buoy IDs: ['44095', '41025']


In [3]:
def find_site_index(kp_coeffs, target_x, target_y, tolerance=1.0):
    """
    Find the site index in kp_coeffs that matches the target coordinates.

    Parameters:
    -----------
    kp_coeffs : xarray.Dataset
        The kp coefficients dataset
    target_x : float
        Target coord_x coordinate
    target_y : float
        Target coord_y coordinate
    tolerance : float
        Tolerance for coordinate matching

    Returns:
    --------
    int
        The site index that matches the coordinates
    """
    # Get all site coordinates
    site_x = kp_coeffs.coord_x.values
    site_y = kp_coeffs.coord_y.values

    # Calculate distances to all sites
    distances = np.sqrt((site_x - target_x) ** 2 + (site_y - target_y) ** 2)

    # Find the site with minimum distance
    site_index = np.argmin(distances)

    # Check if the closest site is within tolerance
    if distances[site_index] > tolerance:
        print(f"Warning: Closest site is {distances[site_index]:.2f} away")

    return site_index


def load_buoy_data(buoy_id, year=None, drop_all_na=False):
    """
    Load buoy data and return both the wave data and location coordinates.
    Modified to handle missing variables gracefully.

    Parameters:
    -----------
    buoy_id : str
        The buoy ID (e.g., '41036')
    year : str, optional
        The year to filter data for (default: None)
    drop_all_na : bool, optional
        If True, drop rows where ALL variables are NaN. If False, only drop rows
        where variables that exist are all NaN (default: False)

    Returns:
    --------
    tuple
        (buoy_waves, buoy_location, kp_coeffs, site_index) or None if loading fails
    """
    try:
        # Load buoy data
        buoy_file = f"../inputs/buoy_data/buoy_{buoy_id}_bulk_parameters.pkl"
        if not os.path.exists(buoy_file):
            print(f"Warning: Buoy file not found: {buoy_file}")
            return None
            
        buoy_waves = pd.read_pickle(buoy_file).sort_index()
      
        # Filter data up to 2023-12-31
        buoy_waves = buoy_waves[buoy_waves.index <= '2023-12-31']
        
        if year:
            buoy_waves = buoy_waves.loc[year]
        
        # Check which columns exist
        available_cols = []
        if "Hs_Buoy" in buoy_waves.columns:
            available_cols.append("Hs_Buoy")
        if "Tp_Buoy" in buoy_waves.columns:
            available_cols.append("Tp_Buoy")
        if "Dir_Buoy" in buoy_waves.columns:
            available_cols.append("Dir_Buoy")
        
        if len(available_cols) == 0:
            print(f"Warning: No valid wave parameters found for buoy {buoy_id}")
            return None
        
        # Only drop rows where ALL available variables are NaN
        # Keep rows that have at least one valid value (even if others are NaN)
        mask = buoy_waves[available_cols].isna().all(axis=1)
        buoy_waves = buoy_waves[~mask]
        
        if len(buoy_waves) == 0:
            print(f"Warning: No valid data after filtering for buoy {buoy_id}")
            return None
        
        # Print info about NaN values for debugging
        nan_counts = {}
        for col in available_cols:
            nan_count = buoy_waves[col].isna().sum()
            nan_counts[col] = nan_count
        if any(nan_counts.values()):
            print(f"NaN counts for buoy {buoy_id}: {nan_counts}")
        
        buoy_location = buoys[buoy_id]

        # Load kp coefficients
        kp_coeffs = xr.open_dataset("outputs/kp_coefficients.nc")
        site_index = find_site_index(kp_coeffs, buoy_location[0], buoy_location[1])
        kp_coeffs = kp_coeffs.isel(site=[site_index])

        return buoy_waves, buoy_location, kp_coeffs, site_index
    
    except Exception as e:
        print(f"Error loading buoy {buoy_id}: {e}")
        return None


def save_validation_csv_with_interpolation(
    buoy_data: pd.DataFrame,
    binwaves_hs: np.ndarray,
    binwaves_tp: np.ndarray,
    binwaves_dpm: np.ndarray,
    buoy_id: str,
    save_path: str = "outputs",
    filename_prefix: str = "buoy",
    target_resolution: str = "1D",
):
    """
    Save validation data to CSV files, creating hourly data and interpolating to target resolution.
    Modified to handle missing variables.
    
    Parameters:
    -----------
    buoy_data : pd.DataFrame
        Buoy wave data with columns: Hs_Buoy, Tp_Buoy, Dir_Buoy (may be missing)
    binwaves_hs : np.ndarray
        BinWaves significant wave height time series
    binwaves_tp : np.ndarray
        BinWaves peak period time series
    binwaves_dpm : np.ndarray
        BinWaves mean direction time series
    buoy_id : str
        Buoy ID for filename
    save_path : str, optional
        Directory to save the CSV files (default: "outputs")
    filename_prefix : str, optional
        Prefix for the filename (default: "buoy")
    target_resolution : str, optional
        Target resolution for interpolation. Options: "3h", "6h", "1D"/"daily" (default: "1D")
    
    Returns:
    --------
    tuple
        (hourly_filepath, interpolated_filepath) - Paths to both saved CSV files
    """
    print(f"Creating validation data with interpolation to {target_resolution}...")

    # Create directory if it doesn't exist
    os.makedirs(save_path, exist_ok=True)

    # Round the datetime index to the nearest hour to align with the hourly grid
    buoy_data = buoy_data.copy()
    buoy_data.index = pd.to_datetime(buoy_data.index).round('1h')
    # Aggregate duplicate timestamps by taking the mean
    buoy_data = buoy_data.groupby(buoy_data.index).mean()

    # Build DataFrame with available columns
    # Handle NaNs properly - keep them in the data
    data_dict = {'datetime': buoy_data.index}
    
    if 'Hs_Buoy' in buoy_data.columns:
        data_dict['Hs_Buoy'] = buoy_data['Hs_Buoy'].values
        # Ensure binwaves_hs has the same length and handle NaNs
        if len(binwaves_hs) != len(buoy_data):
            print(f"Warning: Length mismatch for Hs - buoy: {len(buoy_data)}, binwaves: {len(binwaves_hs)}")
        data_dict['Hs_BinWaves'] = binwaves_hs[:len(buoy_data)] if len(binwaves_hs) >= len(buoy_data) else np.pad(binwaves_hs, (0, len(buoy_data) - len(binwaves_hs)), constant_values=np.nan)
    
    if 'Tp_Buoy' in buoy_data.columns:
        data_dict['Tp_Buoy'] = buoy_data['Tp_Buoy'].values
        # Ensure binwaves_tp has the same length and handle NaNs
        if len(binwaves_tp) != len(buoy_data):
            print(f"Warning: Length mismatch for Tp - buoy: {len(buoy_data)}, binwaves: {len(binwaves_tp)}")
        data_dict['Tp_BinWaves'] = binwaves_tp[:len(buoy_data)] if len(binwaves_tp) >= len(buoy_data) else np.pad(binwaves_tp, (0, len(buoy_data) - len(binwaves_tp)), constant_values=np.nan)
    
    if 'Dir_Buoy' in buoy_data.columns:
        data_dict['Dir_Buoy'] = buoy_data['Dir_Buoy'].values
        # Ensure binwaves_dpm has the same length and handle NaNs
        if len(binwaves_dpm) != len(buoy_data):
            print(f"Warning: Length mismatch for Dir - buoy: {len(buoy_data)}, binwaves: {len(binwaves_dpm)}")
        data_dict['Dir_BinWaves'] = binwaves_dpm[:len(buoy_data)] if len(binwaves_dpm) >= len(buoy_data) else np.pad(binwaves_dpm, (0, len(buoy_data) - len(binwaves_dpm)), constant_values=np.nan)

    # Create hourly DataFrame
    hourly_df = pd.DataFrame(data_dict)
    hourly_df.set_index('datetime', inplace=True)

    # Create regular hourly time index for interpolation
    start_time = hourly_df.index.min()
    end_time = hourly_df.index.max()

    # Create hourly time index
    hourly_times = pd.date_range(start=start_time, end=end_time, freq='1h')

    # Reindex to hourly and interpolate, but only for small gaps (e.g., up to 3 hours)
    hourly_df_interpolated = hourly_df.reindex(hourly_times).interpolate(
        method='linear', limit=3, limit_direction='both', limit_area='inside'
    )

    # Create target resolution time index
    if target_resolution == "1D" or target_resolution == "daily":
        target_times = pd.date_range(start=start_time, end=end_time, freq='1D')
    elif target_resolution == "6h":
        target_times = pd.date_range(start=start_time, end=end_time, freq='6h')
    else:
        # Default to 1D if not specified
        target_times = pd.date_range(start=start_time, end=end_time, freq='1D')

    # Interpolate to target resolution
    target_df = hourly_df_interpolated.reindex(target_times).interpolate(method='linear')

    # Save hourly data
    hourly_filename = f"{filename_prefix}_{buoy_id}_1h.csv"
    hourly_filepath = os.path.join(save_path, hourly_filename)
    hourly_df_interpolated.to_csv(hourly_filepath)

    # Save target resolution data
    target_filename = f"{filename_prefix}_{buoy_id}_{target_resolution}.csv"
    target_filepath = os.path.join(save_path, target_filename)
    target_df.to_csv(target_filepath)

    print(f"Hourly data saved as: {hourly_filepath}")
    print(f"Hourly data shape: {hourly_df_interpolated.shape}")
    print(f"{target_resolution} data saved as: {target_filepath}")
    print(f"{target_resolution} data shape: {target_df.shape}")

    return hourly_filepath, target_filepath

print("Helper functions defined")


Helper functions defined


In [4]:
# Load model parameters and prepare offshore spectra transformation
# This only needs to be done once for all buoys
model_parameters = pd.read_csv("CASES/swan_cases.csv").to_dict(orient="list")

# Load kp_coefficients once (we'll subset it per buoy)
kp_coeffs_full = xr.open_dataset("outputs/kp_coefficients.nc")
print(f"Loaded kp_coefficients with {len(kp_coeffs_full.site)} sites")
print("Model parameters loaded")


FileNotFoundError: [Errno 2] No such file or directory: '/vols/abedul/home/grupos/geocean/montanoj/ShoreShop2026/grid3/outputs/kp_coefficients.nc'

In [ ]:
# Process all buoys
successful_buoys = []
failed_buoys = []

for buoy_id in buoys.keys():
    print(f"\n{'='*60}")
    print(f"Processing buoy: {buoy_id}")
    print(f"{'='*60}")
    
    try:
        # Load buoy data
        result = load_buoy_data(buoy_id)
        if result is None:
            print(f"Skipping buoy {buoy_id} - failed to load")
            failed_buoys.append(buoy_id)
            continue
            
        buoy_waves, buoy_location, kp_coeffs, site_index = result
        
        print(f"Buoy {buoy_id} coordinates: {buoy_location[0]}, {buoy_location[1]}")
        print(f"Selected kp_coeffs site coordinates: {kp_coeffs.coord_x.values}, {kp_coeffs.coord_y.values}")
        print(f"Buoy data shape: {buoy_waves.shape}")
        print(f"Available columns: {list(buoy_waves.columns)}")
        
        # Transform offshore spectrum for this buoy's kp_coeffs
        offshore_spectra, offshore_spectra_case = transform_Offshore_spectrum(
            CAWCR_spectrum=cawcr_spectrum,
            subset_parameters=model_parameters,
            available_case_num=kp_coeffs.case_num.values,
            fixed_direction=True,
        )
        
        # Remove duplicates from offshore_spectra_case
        _, unique_idx_offshore_case = np.unique(offshore_spectra_case.time, return_index=True)
        offshore_spectra_case = offshore_spectra_case.isel(time=unique_idx_offshore_case)
        
        # Remove duplicates from offshore_spectra
        _, unique_idx_offshore = np.unique(offshore_spectra.time, return_index=True)
        offshore_spectra = offshore_spectra.isel(time=unique_idx_offshore)
        
        # Make sure buoy_waves has unique indices
        buoy_waves = buoy_waves[~buoy_waves.index.duplicated(keep="first")]
        
        if len(buoy_waves) == 0:
            print(f"Skipping buoy {buoy_id} - no valid data after deduplication")
            failed_buoys.append(buoy_id)
            continue
        
        # Reconstruct onshore spectra
        print("Reconstructing onshore spectra...")
        reconstructed_onshore_spectra = reconstruc_spectra(
            offshore_spectra=offshore_spectra_case.sel(time=buoy_waves.index, method="nearest"),
            kp_coeffs=kp_coeffs,
            chunk_sizes={"time": 24},
            num_workers=15,
        )
        
        # Ensure all datasets have unique time values
        _, unique_idx_recon = np.unique(reconstructed_onshore_spectra.time, return_index=True)
        reconstructed_onshore_spectra = reconstructed_onshore_spectra.isel(time=unique_idx_recon)
        
        # Prepare data for plotting
        binwaves_spec = reconstructed_onshore_spectra.sel(
            time=buoy_waves.index, method="nearest"
        ).rename({"kps": "efth"}).squeeze().spec
        
        # Plot wave series
        print("Creating plots...")
        try:
            plot_wave_series(
                buoy_data=buoy_waves,
                binwaves_data=binwaves_spec,
                offshore_data=offshore_spectra.sel(time=buoy_waves.index, method="nearest").spec,
                times=buoy_waves.index.values,
                save_plots=True,
                save_path="outputs/Figures/",
                title_prefix=f"Wave Validation {buoy_id}",
            )
            print(f"✓ Plots saved for buoy {buoy_id}")
        except Exception as e:
            print(f"Warning: Could not create plots for buoy {buoy_id}: {e}")
        
        # Save CSV files
        print("Saving CSV files...")
        try:
            # Round and group the buoy index
            buoy_waves_rounded = buoy_waves.copy()
            buoy_waves_rounded.index = pd.to_datetime(buoy_waves_rounded.index).round('1h')
            buoy_waves_rounded = buoy_waves_rounded.groupby(buoy_waves_rounded.index).mean()
            
            # Extract binwaves data
            binwaves_spec_rounded = reconstructed_onshore_spectra.sel(
                time=buoy_waves_rounded.index, method="nearest"
            ).rename({"kps": "efth"}).squeeze().spec
            
            binwaves_hs = binwaves_spec_rounded.hs().values
            binwaves_tp = binwaves_spec_rounded.tp().values
            binwaves_dpm = binwaves_spec_rounded.dpm().values
            
            # Save CSV files
            hourly_file, daily_file = save_validation_csv_with_interpolation(
                buoy_data=buoy_waves_rounded,
                binwaves_hs=binwaves_hs,
                binwaves_tp=binwaves_tp,
                binwaves_dpm=binwaves_dpm,
                buoy_id=buoy_id,
                save_path="outputs/time_series/",
                filename_prefix="buoy_validation",
                target_resolution="1d"
            )
            print(f"✓ CSV files saved for buoy {buoy_id}")
        except Exception as e:
            print(f"Warning: Could not save CSV files for buoy {buoy_id}: {e}")
        
        successful_buoys.append(buoy_id)
        print(f"✓ Successfully processed buoy {buoy_id}")
        
    except Exception as e:
        print(f"✗ Error processing buoy {buoy_id}: {e}")
        import traceback
        traceback.print_exc()
        failed_buoys.append(buoy_id)

print(f"\n{'='*60}")
print("Processing complete!")
print(f"{'='*60}")
print(f"Successfully processed: {len(successful_buoys)} buoys")
print(f"Successful buoys: {successful_buoys}")
if failed_buoys:
    print(f"\nFailed buoys: {len(failed_buoys)}")
    print(f"Failed buoy IDs: {failed_buoys}")
